# 02 — Data Cleaning

This notebook runs the **end-to-end preprocessing pipeline** that turns the
raw IMDB / TMDB / MovieLens files into a single clean dataset:
``data/processed/movies_clean.csv``.

The actual logic lives in ``src/data/preprocessor.py`` — this notebook is
the **operator UI**: it runs each step, prints before/after row counts, and
proves with numbers that:

1. **No nulls** remain in critical columns.
2. **No duplicate `tconst`** values remain.
3. **All ranges are sane** (year, runtime, ratings).
4. **Weighted rating** is computed and behaves like a bell curve.
5. **Memory + runtime** stay within reasonable bounds.

If you already ran ``01_data_collection.ipynb``, the raw files are on disk
and this notebook is fully reproducible.

> **Why weighted rating?** Raw averageRating is *misleading* when vote counts
> are small (a film with 8 votes at 10.0 isn't really better than The Godfather
> at 9.2 with 2M votes). The IMDb-style Bayesian weighted rating shrinks
> low-vote-count films toward the global mean, giving us a metric we can
> actually rank by.

## 0. Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import time, tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import settings, setup_logging
from src.data.preprocessor import MovieDataPreprocessor

setup_logging()

import logging
logger = logging.getLogger("02_data_cleaning")

# Sanity: make sure Step 2 ran first
required = [
    settings.PATHS.RAW_DATA / "title.basics.tsv.gz",
    settings.PATHS.RAW_DATA / "title.ratings.tsv.gz",
    settings.PATHS.RAW_DATA / "title.crew.tsv.gz",
    settings.PATHS.RAW_DATA / "name.basics.tsv.gz",
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Run 01_data_collection.ipynb first — missing: {missing}"
    )
print("All raw IMDB files present:")
for p in required:
    print(f"  {p.name:<28} {p.stat().st_size/1e6:7.1f} MB")

All raw IMDB files present:
  title.basics.tsv.gz            221.6 MB
  title.ratings.tsv.gz             8.4 MB
  title.crew.tsv.gz               81.5 MB
  name.basics.tsv.gz             303.3 MB


## 1. Run the full pipeline with timing & memory tracking

Every step in :class:`MovieDataPreprocessor` logs ``before -> after`` row
counts. Below we instantiate the preprocessor, run the whole pipeline, and
record how long each step takes plus peak memory usage.

In [ ]:
tracemalloc.start()
t0 = time.perf_counter()

prep = MovieDataPreprocessor()
# `with_tmdb=False` is the default for first runs — set True after you've
# populated the TMDB cache (Step 2's optional sample, or a longer fetch run).
prep.run(with_tmdb=True, save=True)

elapsed = time.perf_counter() - t0
_, mem_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\nPipeline finished in {elapsed:.1f}s, peak memory {mem_peak/1e6:.0f} MB")
print(f"Final clean dataset: {len(prep.df_):,} rows × {prep.df_.shape[1]} cols")

2026-05-13 21:36:43 | INFO     | src.data.preprocessor:154 | Loading title.basics.tsv.gz


## 2. Step-by-step audit — proof of what was dropped and why

In [ ]:
audit = prep.quality_report()
audit

## 3. Null report on the FINAL dataset

These five columns are **required** by every downstream model:
``tconst``, ``primaryTitle``, ``startYear``, ``runtimeMinutes``,
``averageRating``, ``numVotes``, ``weighted_rating``. Their ``null_count``
must be **exactly zero** — anything else is a bug.

In [ ]:
null_rep = prep.null_report()
# Highlight critical columns
critical = ["tconst", "primaryTitle", "startYear", "runtimeMinutes",
            "averageRating", "numVotes", "weighted_rating"]
print("--- CRITICAL COLUMNS (null_count MUST be 0) ---")
print(null_rep.loc[null_rep.index.isin(critical)].to_string())
print("\n--- OPTIONAL COLUMNS (NaN allowed) ---")
print(null_rep.loc[~null_rep.index.isin(critical)].to_string())

## 4. Duplicate check

In [ ]:
n_dup_tconst = int(prep.df_['tconst'].duplicated().sum())
print(f"Duplicate tconst values in final dataset: {n_dup_tconst}")
assert n_dup_tconst == 0, "PIPELINE BUG: duplicates were not removed"
print("✓ No duplicates")

## 5. Sanity ranges

Every numeric column has a plausible range. If anything is way out of
expected bounds (e.g. a film from year 9999, or a runtime of 1500 minutes),
there's a bug upstream we need to investigate.

In [ ]:
df = prep.df_
sanity = pd.DataFrame({
    "min": [df['startYear'].min(),     df['runtimeMinutes'].min(),
            df['averageRating'].min(), df['numVotes'].min(),
            df['weighted_rating'].min()],
    "max": [df['startYear'].max(),     df['runtimeMinutes'].max(),
            df['averageRating'].max(), df['numVotes'].max(),
            df['weighted_rating'].max()],
}, index=['startYear','runtimeMinutes','averageRating','numVotes','weighted_rating'])
sanity

## 6. Weighted rating — visual proof of the Bayesian shrinkage

Two plots tell the story:

* **Histogram** — weighted rating should be ~bell-curved around the global mean.
* **Scatter** — for low-vote-count films, weighted rating is pulled toward
  the global mean; for high-vote-count films, it stays close to the raw
  ``averageRating``.

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) histogram
axes[0].hist(df['weighted_rating'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df['weighted_rating'].mean(), color='red', linestyle='--',
                label=f"mean = {df['weighted_rating'].mean():.2f}")
axes[0].set_xlabel('Weighted rating')
axes[0].set_ylabel('Number of movies')
axes[0].set_title('Distribution of weighted ratings')
axes[0].legend()

# (b) shrinkage scatter: numVotes (log) vs (averageRating - weighted_rating)
delta = df['averageRating'] - df['weighted_rating']
axes[1].scatter(df['numVotes'], delta, alpha=0.25, s=8, color='darkorange')
axes[1].set_xscale('log')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('numVotes (log scale)')
axes[1].set_ylabel('averageRating − weighted_rating')
axes[1].set_title('Bayesian shrinkage: low-vote films are pulled toward the mean')

plt.tight_layout()
fig_path = settings.PATHS.FIGURES / '02_weighted_rating_proof.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved {fig_path}")

## 7. Top 20 films by weighted rating

This is the most direct sanity check: does the top of our ranked list look
like a sensible "best films of all time" selection? If yes, the weighted
rating is doing its job.

In [ ]:
top20 = df.nlargest(20, 'weighted_rating')[
    ['tconst','primaryTitle','startYear','averageRating',
     'numVotes','weighted_rating','director_name']
].reset_index(drop=True)
top20

## 8. Genre distribution

A quick look at how often each genre appears. (Each film can have multiple
genres, so the totals exceed the number of films.)

In [ ]:
genre_counts = df['genres_list'].explode().value_counts()
print(genre_counts.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
genre_counts.head(20).plot.barh(ax=ax, color='teal')
ax.invert_yaxis()
ax.set_xlabel('Number of films')
ax.set_title('Top 20 genres (films can belong to multiple genres)')
plt.tight_layout()
plt.savefig(settings.PATHS.FIGURES / '02_genre_counts.png', dpi=130, bbox_inches='tight')
plt.show()

## 9. Year & runtime sanity plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['startYear'], bins=range(int(df['startYear'].min()),
                                         int(df['startYear'].max())+2),
             color='indigo', edgecolor='white')
axes[0].set_xlabel('Release year')
axes[0].set_ylabel('Number of films')
axes[0].set_title('Films per year')

axes[1].hist(df['runtimeMinutes'], bins=50, color='crimson', edgecolor='white')
axes[1].set_xlabel('Runtime (minutes)')
axes[1].set_ylabel('Number of films')
axes[1].set_title('Runtime distribution')

plt.tight_layout()
plt.savefig(settings.PATHS.FIGURES / '02_year_runtime.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Save audit log alongside the clean CSV

For reproducibility we persist the full audit table so any reviewer can
verify exactly how many rows were dropped at every step.

In [ ]:
audit_path = settings.PATHS.PROCESSED_DATA / '02_cleaning_audit.csv'
prep.quality_report().to_csv(audit_path, index=False)
print(f"Wrote {audit_path}")
print(f"Clean dataset:  {settings.PATHS.MOVIES_CLEAN}")
print(f"  size:   {settings.PATHS.MOVIES_CLEAN.stat().st_size/1e6:.1f} MB")
print(f"  rows:   {len(df):,}")
print(f"  cols:   {df.shape[1]}")

## ✅ Step 3 outcomes

* **`movies_clean.csv`** written to `data/processed/`. This is the canonical
  input for everything that follows (EDA, content-based recommender, hybrid
  engine).
* **Cleaning audit** persisted to `02_cleaning_audit.csv` — a tabular
  before/after row count for every step.
* **Proof plots** saved to `reports/figures/02_*.png`.

Next: **Step 4 — Exploratory Data Analysis** (`03_eda_general.ipynb`).